In [5]:
import os
print(os.getcwd())
print(os.listdir('.'))

C:\Users\Lenovo\OneDrive\Desktop\Brand Sentiment Analyzer
['.git', '.gitignore', '.ipynb_checkpoints', '02_cleaning.ipynb', 'app', 'data', 'eshal', 'models', 'notebooks', 'reports', 'requirements.txt', 'src']


In [6]:
import pandas as pd
import numpy as np

# Removed the '../' because your notebook is already in the main project folder
df = pd.read_csv('data/raw/training.1600000.processed.noemoticon.csv',
                 encoding='latin-1',
                 names=['sentiment', 'id', 'date', 'query', 'user', 'text'])

# See the first 5 rows
df.head()

,sentiment,id,date,query,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [7]:
#STEP 1: REMAP SENTIMENT LABELS

## Convert 4 -> 1 so labels are 0 (negative) and 1 (positive)
df['sentiment'] = df['sentiment'].map({0: 0, 4: 1})

In [8]:
#STEP 2: KEEP ONLY WHAT YOU NEED

## We only need text and sentiment
df = df[['text', 'sentiment']].copy()

In [9]:
#STEP 3: REMOVE DUPLICATES

df.drop_duplicates(subset='text', inplace=True)
print('After dedup:', df.shape)

After dedup: (1581466, 2)


In [10]:
#STEP 4: WRITING THE CLEANING FUNCTION

import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))

def clean_text(text):
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # 3. Remove @mentions
    text = re.sub(r'@\w+', '', text)
    
    # 4. Remove hashtag symbol but keep word
    text = re.sub(r'#', '', text)
    
    # 5. Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # 6. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 7. Remove extra whitespace
    text = ' '.join(text.split())
    
    # 8. Remove stopwords
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(clean_text)
print(df[['text', 'clean_text']].head())

                                                text  \
0  @switchfoot http://twitpic.com/2y1zl - Awww, t...   
1  is upset that he can't update his Facebook by ...   
2  @Kenichan I dived many times for the ball. Man...   
3    my whole body feels itchy and like its on fire    
4  @nationwideclass no, it's not behaving at all....   

                                          clean_text  
0      thats bummer shoulda got david carr third day  
1  upset cant update facebook texting might cry r...  
2  dived many times ball managed save rest go bounds  
3                   whole body feels itchy like fire  
4                           behaving im mad cant see  


In [11]:
#STEP 5: REMOVE EMPTY ROWS AFTER CLEANING

df = df[df['clean_text'].str.strip() != '']
df.dropna(subset=['clean_text'], inplace=True)
print('Final shape:', df.shape)

Final shape: (1574472, 3)


In [12]:
#STEP 6: SAMPLE FOR SPEED (OPTIONAL)

# Take 100,000 balanced samples (50k positive, 50k negative)
df_pos = df[df['sentiment'] == 1].sample(50000, random_state=42)
df_neg = df[df['sentiment'] == 0].sample(50000, random_state=42)
df = pd.concat([df_pos, df_neg]).sample(frac=1, random_state=42).reset_index(drop=True)
print('Sampled shape:', df.shape)

Sampled shape: (100000, 3)


In [13]:
#STEP 7: SAVE THE CLEAN DATA

df.to_csv('data/processed/clean_data.csv', index=False)
print('Saved to data/processed/clean_data.csv')

Saved to data/processed/clean_data.csv
